# Limpieza de datos – Reto Venture One (Ficha de mercado y entorno)

Este cuaderno limpia los tres CSV del reto:

| Archivo | Qué contiene |
|---|---|
| `anuncios_muestra.csv` | Anuncios de inmuebles (comparables): área, precio, estado, etc. |
| `inmuebles_prueba.csv` | Los 11 inmuebles que vamos a evaluar, con coordenadas |
| `inmuebles_prueba_fuente.csv` | De dónde salió la dirección / coordenada de cada inmueble |

**Cómo usarlo:** ejecuta las celdas **una por una, de arriba hacia abajo** (`Shift + Enter`).
Lee el texto antes de cada celda: explica *qué* hacemos y *por qué*.

**Al final obtendrás**, en la carpeta `datos/limpios/`:
- `anuncios_limpios.csv` – comparables sin duplicados, con tipos correctos y precio por m²
- `inmuebles_limpios.csv` – inmuebles de prueba unidos con su fuente y validados
- `reporte_calidad.csv` – registro de **qué se corrigió, qué falta y qué es sospechoso**
  (la ficha del reto dice que hoy "no se deja registro de los datos faltantes": este archivo lo resuelve)

## Paso 1. Importar librerías

- **pandas**: la librería principal para trabajar con tablas (DataFrames).
- **numpy**: operaciones numéricas; la usamos para `NaN` (valor faltante).
- **unicodedata**: viene con Python; sirve para quitar tildes (`Bomboná` → `Bombona`).
- **pathlib**: para manejar rutas de carpetas sin problemas entre Windows y Mac.

In [61]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# Mostrar todas las columnas al imprimir una tabla
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

CARPETA_CRUDOS = Path("datos") / "originales"     # los 3 CSV
CARPETA_GEO = CARPETA_CRUDOS / "geo"               # los 2 GeoJSON
CARPETA_LIMPIOS = Path("datos") / "limpios"       # aquí se guardan los resultados
CARPETA_LIMPIOS.mkdir(exist_ok=True)   # crea la carpeta si no existe

print("pandas versión:", pd.__version__)

pandas versión: 3.1.1


## Paso 2. Cargar los archivos

Dos detalles importantes de estos CSV:

1. **El separador es `;`** (punto y coma), no coma. Si no lo indicas, pandas mete todo en una sola columna.
2. Leemos todo como **texto** (`dtype=str`). Así pandas no "adivina" tipos y no se pierde nada
   (por ejemplo, un `?` en una columna numérica). Luego convertimos cada columna a propósito.

`keep_default_na=False` hace que las celdas vacías lleguen como texto vacío `""`; las convertimos a faltante nosotros mismos para tener control.

In [62]:
def cargar_csv(nombre):
    ruta = CARPETA_CRUDOS / nombre

    # Detectar el separador: miramos la primera línea (encabezados)
    with open(ruta, encoding="utf-8-sig") as f:
        primera_linea = f.readline()
    separador = ";" if primera_linea.count(";") > primera_linea.count(",") else ","

    df = pd.read_csv(ruta, sep=separador, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas (separador '{separador}')")
    return df

anuncios_raw = cargar_csv("anuncios_muestra.csv")
inmuebles_raw = cargar_csv("inmuebles_prueba.csv")
fuente_raw = cargar_csv("inmuebles_prueba_fuente.csv")

# Trabajamos sobre copias: el original queda intacto para comparar al final
anuncios = anuncios_raw.copy()
inmuebles = inmuebles_raw.copy()
fuente = fuente_raw.copy()

anuncios_muestra.csv: 117 filas x 14 columnas (separador ',')
inmuebles_prueba.csv: 11 filas x 13 columnas (separador ',')
inmuebles_prueba_fuente.csv: 11 filas x 6 columnas (separador ',')


## Paso 3. Explorar antes de limpiar (diagnóstico)

**Nunca limpies a ciegas.** Primero mira los datos. Estas son las preguntas clave:

- ¿Cómo se ven las primeras filas? → `head()`
- ¿Cuántas celdas vacías hay por columna?
- ¿Qué valores distintos tiene cada columna de texto? → `value_counts()`

In [63]:
anuncios.head(10)

,id_anuncio,fuente,fecha,zona,municipio,barrio,area_m2,precio,habitaciones,banos,parqueadero,estado,tipo_precio,origen
0,fincaraiz:10710482,fincaraiz,,castellana_laureles,medellin,la castellana,94,610000000,3,2,1,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:2
1,fincaraiz:193287983,fincaraiz,,castellana_laureles,medellin,la castellana,99,690000000,3,2,2,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:3
2,fincaraiz:192780858,fincaraiz,,castellana_laureles,medellin,laureles,92,497000000,3,2,-,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:4
3,fincaraiz:192971383,fincaraiz,,castellana_laureles,medellin,la castellana,91,590000000,3,3,1,HABITABLE POR REMODELAR,pedido,factibilidad_castellana.xlsx:SECTORIAL:5
4,fincaraiz:193663348,fincaraiz,,castellana_laureles,medellin,santa teresita,89,580000000,4,3,1,HABITABLE POR REMODELAR,pedido,factibilidad_castellana.xlsx:SECTORIAL:6
5,fincaraiz:193632316,fincaraiz,,castellana_laureles,medellin,laureles,90,550000000,3,2,?,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:7
6,fincaraiz:192987663,fincaraiz,,castellana_laureles,medellin,laureles,94,720000000,3,2,1,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:8
7,fincaraiz:193440669,fincaraiz,,castellana_laureles,medellin,la castellana,100,870000000,3,2,1,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:9
8,fincaraiz:193287983,fincaraiz,,castellana_laureles,medellin,la castellana,99,690000000,3,2,2,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:10
9,fincaraiz:10710482,fincaraiz,,castellana_laureles,medellin,la castellana,94,610000000,3,2,1,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:11


In [64]:
# Celdas vacías (texto "") por columna
(anuncios == "").sum()

id_anuncio       50
fuente            0
fecha           117
zona              0
municipio         0
barrio           25
area_m2           1
precio            0
habitaciones      5
banos             9
parqueadero      82
estado           25
tipo_precio       0
origen            0
dtype: int64

In [65]:
# Valores distintos de las columnas "categóricas"
for col in ["fuente", "zona", "municipio", "parqueadero", "estado", "tipo_precio"]:
    print(f"\n=== {col} ===")
    print(anuncios[col].value_counts(dropna=False))


=== fuente ===
fuente
fincaraiz                 56
facebook                  31
aviso_calle               18
arrendamientos_santafe     6
metrocuadrado              4
ciencuadras                1
otra                       1
Name: count, dtype: int64

=== zona ===
zona
buenos_aires_el_salvador    82
ditaires_itagui             19
castellana_laureles         16
Name: count, dtype: int64

=== municipio ===
municipio
medellin    98
itagui      18
envigado     1
Name: count, dtype: int64

=== parqueadero ===
parqueadero
     82
1    28
2     3
?     2
-     1
0     1
Name: count, dtype: int64

=== estado ===
estado
                              25
Remodelar                     20
OK                            16
REMODELADA                    13
REMODELADO                    10
Vivir                          9
HABITABLE POR REMODELAR        6
remodelar                      5
SIN REMODELAR (OBRA NEGRA)     3
Reformar                       3
SIN REMODELAR                  2
Ok               

### ¿Qué encontramos? (problemas a resolver)

| # | Problema | Ejemplo |
|---|---|---|
| 1 | `fecha` está **vacía en todas** las filas | — |
| 2 | Muchos anuncios **sin `id_anuncio`** (Facebook, aviso de calle) | fila 37 |
| 3 | Un id con un carácter basura | `fincaraiz:191722056#` |
| 4 | **Duplicados**: el mismo anuncio aparece en varias hojas del Excel original | `fincaraiz:7387557` aparece 3 veces |
| 5 | `parqueadero` con símbolos que no son números | `-`, `?` |
| 6 | `estado` escrito de **15 formas** distintas | `REMODELADO`, `REMODELADA ?`, `vivir`, `OK`… |
| 7 | `barrio` sin estandarizar: mayúsculas, errores, notas | `salvadror/palmas`, `El Salvador (arriba de gimnasio…)` |
| 8 | Faltan área, habitaciones o baños en algunas filas | fila 47 sin área |
| 9 | Valores numéricos que hay que **convertir de texto a número** | `70.4`, `610000000` |
| 10 | Posibles **datos atípicos** (precio/m² muy alto o muy bajo) | 43 m² con 4 habitaciones |

## Paso 4. Crear el "registro de calidad"

Antes de cambiar nada, creamos una lista donde **anotamos cada problema** que encontremos
(archivo, fila, columna, qué pasó). Al final la guardamos como `reporte_calidad.csv`.
Así el equipo de inversión sabe qué datos son confiables.

In [66]:
registro = []   # lista de diccionarios; al final se vuelve DataFrame

def anotar(archivo, fila_id, columna, problema, valor_original="", accion=""):
    registro.append({
        "archivo": archivo,
        "fila_id": fila_id,
        "columna": columna,
        "problema": problema,
        "valor_original": valor_original,
        "accion": accion,
    })

# Agregamos un número de fila para poder rastrear cada registro hasta el original
anuncios.insert(0, "fila_original", range(2, len(anuncios) + 2))  # +2: fila 1 es el encabezado

## Paso 5. Limpieza de texto general

Funciones pequeñas y reutilizables:

- `quitar_tildes("Bomboná")` → `"Bombona"`
- `normalizar_texto("  El  Salvador ")` → `"el salvador"` (minúsculas, sin tildes, sin espacios dobles)

Luego quitamos espacios sobrantes en **todas** las columnas y convertimos `""` en `NaN` (faltante real).

In [67]:
def quitar_tildes(texto):
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))

def normalizar_texto(valor):
    if pd.isna(valor):
        return np.nan
    valor = quitar_tildes(str(valor)).lower().strip()
    valor = " ".join(valor.split())        # colapsa espacios dobles
    return valor if valor else np.nan

def limpiar_espacios_y_vacios(df):
    df = df.copy()
    for col in df.columns:
        # funciona en pandas 2 (texto = object) y pandas 3 (texto = str)
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.strip().replace("", np.nan)
    return df

anuncios = limpiar_espacios_y_vacios(anuncios)
inmuebles = limpiar_espacios_y_vacios(inmuebles)
fuente = limpiar_espacios_y_vacios(fuente)

anuncios.isna().sum()

fila_original      0
id_anuncio        50
fuente             0
fecha            117
zona               0
municipio          0
barrio            25
area_m2            1
precio             0
habitaciones       5
banos              9
parqueadero       82
estado            25
tipo_precio        0
origen             0
dtype: int64

## Paso 6. Columnas vacías: `fecha`

`fecha` no tiene ni un dato. **No la borramos** (el diccionario del reto la menciona y en el futuro puede venir llena),
pero la convertimos a tipo fecha y dejamos constancia en el registro.

In [68]:
anuncios["fecha"] = pd.to_datetime(anuncios["fecha"], errors="coerce", dayfirst=True)
if anuncios["fecha"].isna().all():
    anotar("anuncios_muestra.csv", "TODAS", "fecha", "columna sin datos", "", "se conserva vacía")
    print("⚠ La columna fecha está 100% vacía")

⚠ La columna fecha está 100% vacía


## Paso 7. Limpiar `id_anuncio`

- Quitamos caracteres que no deberían estar (como `#`). Un id válido tiene la forma `fuente:codigo`
  con letras, números y guiones.
- Revisamos que el prefijo del id coincida con la columna `fuente`.

In [69]:
id_antes = anuncios["id_anuncio"].copy()
anuncios["id_anuncio"] = anuncios["id_anuncio"].str.replace(r"[^A-Za-z0-9_:\-]", "", regex=True)

cambiados = anuncios[id_antes.notna() & (id_antes != anuncios["id_anuncio"])]
for _, f in cambiados.iterrows():
    anotar("anuncios_muestra.csv", f["fila_original"], "id_anuncio",
           "caracteres no válidos", id_antes[_], f"corregido a {f['id_anuncio']}")
print("ids corregidos:\n", cambiados[["fila_original", "id_anuncio"]])

# ¿El prefijo del id coincide con la fuente?
prefijo = anuncios["id_anuncio"].str.split(":").str[0]
no_coincide = anuncios[prefijo.notna() & (prefijo != anuncios["fuente"])]
print("ids cuyo prefijo no coincide con la fuente:", len(no_coincide))

sin_id = anuncios["id_anuncio"].isna().sum()
print("anuncios sin id:", sin_id)

ids corregidos:
     fila_original           id_anuncio
52             54  fincaraiz:191722056
ids cuyo prefijo no coincide con la fuente: 0
anuncios sin id: 50


## Paso 8. Convertir columnas numéricas

Usamos `pd.to_numeric(..., errors="coerce")`: lo que **no** sea número (`-`, `?`) se vuelve `NaN`.
Antes de convertir, anotamos qué valores raros había para no perder esa información.

In [70]:
columnas_numericas = ["area_m2", "precio", "habitaciones", "banos", "parqueadero"]

for col in columnas_numericas:
    original = anuncios[col].copy()
    anuncios[col] = pd.to_numeric(anuncios[col].str.replace(",", ".", regex=False), errors="coerce")
    raros = anuncios[original.notna() & anuncios[col].isna()]
    for i, f in raros.iterrows():
        anotar("anuncios_muestra.csv", f["fila_original"], col,
               "valor no numérico", original[i], "convertido a vacío")

# Habitaciones, baños y parqueadero son enteros. "Int64" (con I mayúscula) permite enteros CON faltantes
for col in ["habitaciones", "banos", "parqueadero"]:
    anuncios[col] = anuncios[col].astype("Int64")

anuncios[columnas_numericas].describe()

,area_m2,precio,habitaciones,banos,parqueadero
count,116.000000,1.170000e+02,112.0,108.0,32.0
mean,80.739138,3.775547e+08,2.75,1.722222,1.0625
std,38.131195,1.905670e+08,0.953892,0.57735,0.353553
min,37.000000,1.200000e+08,1.0,1.0,0.0
25%,61.750000,2.450000e+08,2.0,1.0,1.0
50%,70.400000,3.200000e+08,3.0,2.0,1.0
75%,90.000000,4.600000e+08,3.0,2.0,1.0
max,292.000000,1.000000e+09,9.0,4.0,2.0


## Paso 9. Estandarizar `estado` (la más importante para el reto)

El reto pide precios por m² **según el estado** del inmueble, así que necesitamos pocas categorías claras.
Construimos un **diccionario de mapeo**: a la izquierda cómo viene escrito (normalizado), a la derecha la categoría estándar.

> ⚠ **Revisa este mapeo con el archivo de parámetros del reto** (dice qué estados usar). Interpretaciones que asumimos:
> - `vivir`, `ok`, `remodelado/a` → **para_vivir** (listo para habitar)
> - `habitable por remodelar`, `remodelar`, `reformar`, `sin remodelar` → **por_remodelar**
> - `sin remodelar (obra negra)` → **obra_negra**
>
> Si cambian las categorías, solo modificas este diccionario.

In [71]:
MAPA_ESTADO = {
    "remodelado": "para_vivir",
    "remodelada": "para_vivir",
    "vivir": "para_vivir",
    "ok": "para_vivir",
    "habitable por remodelar": "por_remodelar",
    "remodelar": "por_remodelar",
    "reformar": "por_remodelar",
    "sin remodelar": "por_remodelar",
    "sin remodelar (obra negra)": "obra_negra",
}

def estandarizar_estado(valor):
    texto = normalizar_texto(valor)
    if pd.isna(texto):
        return np.nan, False
    dudoso = "?" in texto                      # "REMODELADA ?" -> el autor no estaba seguro
    texto = texto.replace("?", "").strip()
    return MAPA_ESTADO.get(texto, "SIN_MAPEAR"), dudoso

def aplicar_estado(df, archivo, col_id):
    resultado = df["estado"].apply(estandarizar_estado)
    df["estado_original"] = df["estado"]
    df["estado"] = resultado.str[0]
    df["estado_dudoso"] = resultado.str[1]
    for _, f in df[df["estado_dudoso"]].iterrows():
        anotar(archivo, f[col_id], "estado", "estado marcado con '?'", f["estado_original"], "revisar")
    for _, f in df[df["estado"] == "SIN_MAPEAR"].iterrows():
        anotar(archivo, f[col_id], "estado", "estado no reconocido", f["estado_original"], "agregar al MAPA_ESTADO")
    return df

anuncios = aplicar_estado(anuncios, "anuncios_muestra.csv", "fila_original")

# Tabla de control: cómo quedó cada valor original
anuncios.groupby(["estado_original", "estado"], dropna=False).size()

estado_original             estado       
HABITABLE POR REMODELAR     por_remodelar     6
OK                          para_vivir       16
Ok                          para_vivir        2
REMODELADA                  para_vivir       13
REMODELADA ?                para_vivir        1
REMODELADO                  para_vivir       10
Reformar                    por_remodelar     3
Remodelar                   por_remodelar    20
SIN REMODELAR               por_remodelar     2
SIN REMODELAR (OBRA NEGRA)  obra_negra        3
VIVIR                       para_vivir        1
Vivir                       para_vivir        9
remodelar                   por_remodelar     5
vivir                       para_vivir        1
NaN                         NaN              25
dtype: int64

## Paso 10. Estandarizar `barrio` y `municipio`

1. Normalizamos (minúsculas, sin tildes).
2. Quitamos notas entre paréntesis: `el salvador (diagonal a la iglesia)` → `el salvador`. La nota no se pierde: queda en `barrio_nota`.
3. Corregimos errores de escritura con un diccionario (`salvadror/palmas` → `el salvador`).
4. Revisamos que el municipio sea coherente con la zona.

In [72]:
CORRECCIONES_BARRIO = {
    "salvadror/palmas": "el salvador",   # error de digitación; asumimos El Salvador
    "bombona no1": "bombona no. 1",
}

def limpiar_barrio(valor):
    texto = normalizar_texto(valor)
    if pd.isna(texto):
        return np.nan
    texto = texto.split("(")[0].strip()          # quita lo que está entre paréntesis
    return CORRECCIONES_BARRIO.get(texto, texto)

anuncios["barrio_nota"] = anuncios["barrio"].str.extract(r"\((.*)\)", expand=False)
anuncios["barrio"] = anuncios["barrio"].apply(limpiar_barrio)
anuncios["municipio"] = anuncios["municipio"].apply(normalizar_texto)
anuncios["zona"] = anuncios["zona"].apply(normalizar_texto)
anuncios["fuente"] = anuncios["fuente"].apply(normalizar_texto)

print(anuncios["barrio"].value_counts(dropna=False))

# Coherencia zona ↔ municipio: la zona ditaires_itagui debería ser Itagüí
incoherentes = anuncios[(anuncios["zona"] == "ditaires_itagui") & (anuncios["municipio"] != "itagui")]
for _, f in incoherentes.iterrows():
    anotar("anuncios_muestra.csv", f["fila_original"], "municipio",
           "municipio no coincide con la zona", f["municipio"], "revisar")
incoherentes[["fila_original", "zona", "municipio", "barrio"]]

barrio
el salvador       37
NaN               25
bombona no. 1     17
ditaires          16
la castellana     10
laureles           4
las palmas         3
la milagrosa       2
santa teresita     1
simon bolivar      1
buenos aires       1
Name: count, dtype: int64


,fila_original,zona,municipio,barrio
16,18,ditaires_itagui,envigado,NaN


## Paso 11. Eliminar duplicados

Hay dos tipos:

**A) Anuncios con `id_anuncio`**: si el id se repite, es el mismo anuncio. Pero ojo, a veces una copia está
más completa que otra (ej. `fincaraiz:8146402`: una fila sin habitaciones, otra con 3 hab / 2 baños).
Por eso **nos quedamos con la fila más completa**.

**B) Anuncios sin id** (Facebook, aviso de calle): construimos una "huella" con
`zona + área + precio`. Si dos filas tienen la misma huella, las tratamos como el mismo inmueble.

> Esto es una decisión de negocio razonable, no una verdad absoluta: dos apartamentos distintos *podrían*
> tener la misma área y precio. Por eso todo queda anotado en el registro.

In [73]:
filas_antes = len(anuncios)

# Cuántas columnas tiene llenas cada fila (para elegir la más completa)
anuncios["completitud"] = anuncios[["area_m2", "precio", "habitaciones", "banos",
                                    "parqueadero", "estado", "barrio"]].notna().sum(axis=1)

# Clave de duplicado: el id si existe; si no, la huella
huella = (anuncios["zona"].astype(str) + "|" +
          anuncios["area_m2"].astype(str) + "|" +
          anuncios["precio"].astype(str))
anuncios["clave_duplicado"] = anuncios["id_anuncio"].fillna("SIN_ID|" + huella)

# Cuántas veces aparece cada anuncio y en qué hojas (útil como trazabilidad)
anuncios["veces_repetido"] = anuncios.groupby("clave_duplicado")["clave_duplicado"].transform("size")
anuncios["origenes"] = anuncios.groupby("clave_duplicado")["origen"].transform(lambda s: " | ".join(s))

# Ordenamos: primero la fila más completa, y nos quedamos con la primera de cada clave
anuncios = anuncios.sort_values(["clave_duplicado", "completitud"], ascending=[True, False])
eliminados = anuncios[anuncios.duplicated("clave_duplicado", keep="first")]
anuncios = anuncios.drop_duplicates("clave_duplicado", keep="first")

for _, f in eliminados.iterrows():
    anotar("anuncios_muestra.csv", f["fila_original"], "(fila completa)",
           "duplicado", f["clave_duplicado"], "eliminado (se conservó la copia más completa)")

print(f"Filas antes: {filas_antes} | duplicados eliminados: {len(eliminados)} | filas después: {len(anuncios)}")

Filas antes: 117 | duplicados eliminados: 29 | filas después: 88


### Paso 11b. Posibles duplicados entre portales

El mismo apartamento puede estar publicado en **dos portales** con ids distintos
(ej. `fincaraiz:7387557` y `arrendamientos_santafe:V10746`: los dos son 73 m², $300M, El Salvador).
Como no es seguro, **no los borramos**: los marcamos con `posible_duplicado = True` para revisión.

In [74]:
huella_final = (anuncios["zona"].astype(str) + "|" + anuncios["area_m2"].astype(str) + "|" +
                anuncios["precio"].astype(str))
anuncios["posible_duplicado"] = huella_final.duplicated(keep=False)

for _, f in anuncios[anuncios["posible_duplicado"]].iterrows():
    anotar("anuncios_muestra.csv", f["fila_original"], "(fila completa)",
           "posible duplicado en otro portal (misma zona, área y precio)",
           f["id_anuncio"], "revisar manualmente")

anuncios[anuncios["posible_duplicado"]].sort_values(["area_m2", "precio"])[
    ["fila_original", "id_anuncio", "fuente", "barrio", "area_m2", "precio", "habitaciones"]]

,fila_original,id_anuncio,fuente,barrio,area_m2,precio,habitaciones
62,64,arrendamientos_santafe:V12132,arrendamientos_santafe,el salvador,57.0,225000000,2
54,56,fincaraiz:10766727,fincaraiz,el salvador,57.0,225000000,2
63,65,arrendamientos_santafe:V10746,arrendamientos_santafe,el salvador,73.0,300000000,3
93,95,fincaraiz:7387557,fincaraiz,el salvador,73.0,300000000,3
23,25,fincaraiz:192937244,fincaraiz,NaN,73.0,490000000,3
19,21,fincaraiz:192959138,fincaraiz,ditaires,73.0,490000000,3
21,23,fincaraiz:193131396,fincaraiz,ditaires,73.0,490000000,3


## Paso 12. Datos faltantes en columnas clave

**No inventamos datos** (no rellenamos con promedios), porque esto alimenta decisiones de inversión.
Lo que hacemos es:

- Anotar cada faltante en el registro.
- Marcar con `apto_comparable = False` las filas sin **área o precio** (sin ellos no se puede calcular precio/m²).

In [75]:
for col in ["area_m2", "precio", "habitaciones", "banos", "estado", "barrio"]:
    for _, f in anuncios[anuncios[col].isna()].iterrows():
        anotar("anuncios_muestra.csv", f["fila_original"], col, "dato faltante", "", "se deja vacío")

anuncios["apto_comparable"] = anuncios["area_m2"].notna() & anuncios["precio"].notna()
print("Filas no aptas (sin área o precio):", (~anuncios["apto_comparable"]).sum())

anuncios[["area_m2", "precio", "habitaciones", "banos", "parqueadero", "estado", "barrio"]].isna().sum()

Filas no aptas (sin área o precio): 1


area_m2          1
precio           0
habitaciones     2
banos            5
parqueadero     59
estado          17
barrio          20
dtype: int64

## Paso 13. Calcular precio por m² y detectar atípicos

`precio_m2 = precio / area_m2`

Para detectar valores raros usamos el **método del rango intercuartílico (IQR)**, que es estándar en análisis básico:

- Q1 = percentil 25, Q3 = percentil 75, IQR = Q3 − Q1
- Es atípico si está **por debajo de Q1 − 1.5·IQR** o **por encima de Q3 + 1.5·IQR**

Lo calculamos **por zona**, porque un m² en La Castellana no cuesta lo mismo que en El Salvador.
Además revisamos reglas de sentido común (ej. menos de 12 m² por habitación).

Los atípicos **no se borran**: se marcan para que una persona los revise.

In [76]:
anuncios["precio_m2"] = (anuncios["precio"] / anuncios["area_m2"]).round(0)

def marcar_atipicos(grupo):
    q1, q3 = grupo.quantile(0.25), grupo.quantile(0.75)
    iqr = q3 - q1
    return (grupo < q1 - 1.5 * iqr) | (grupo > q3 + 1.5 * iqr)

anuncios["atipico_precio_m2"] = (anuncios.groupby("zona")["precio_m2"]
                                 .transform(marcar_atipicos).fillna(False).astype(bool))

# Regla de sentido común: área por habitación muy pequeña
anuncios["area_sospechosa"] = (anuncios["area_m2"] / anuncios["habitaciones"].astype(float)) < 12

for _, f in anuncios[anuncios["atipico_precio_m2"]].iterrows():
    anotar("anuncios_muestra.csv", f["fila_original"], "precio_m2", "precio/m² atípico en su zona",
           f["precio_m2"], "revisar")
for _, f in anuncios[anuncios["area_sospechosa"]].iterrows():
    anotar("anuncios_muestra.csv", f["fila_original"], "area_m2", "menos de 12 m² por habitación",
           f"{f['area_m2']} m², {f['habitaciones']} hab", "revisar")

anuncios[anuncios["atipico_precio_m2"] | anuncios["area_sospechosa"]][
    ["fila_original", "zona", "barrio", "area_m2", "habitaciones", "precio", "precio_m2"]]

,fila_original,zona,barrio,area_m2,habitaciones,precio,precio_m2
64,66,buenos_aires_el_salvador,el salvador,43.0,4,290000000,6744186.0
33,35,ditaires_itagui,ditaires,101.0,3,910000000,9009901.0


In [77]:
# Resumen por zona y estado: ¡esto ya empieza a responder la pregunta del reto!
(anuncios[anuncios["apto_comparable"] & ~anuncios["atipico_precio_m2"]]
 .groupby(["zona", "estado"])["precio_m2"]
 .agg(cantidad="count", mediana="median", minimo="min", maximo="max")
 .round(0))

cantidad    mediana     minimo     maximo
zona                     estado                                                  
buenos_aires_el_salvador para_vivir           17  3909091.0  2661972.0  6615385.0
                         por_remodelar        22  4407363.0  2247191.0  6744186.0
castellana_laureles      para_vivir            7  6969697.0  5402174.0  9111111.0
                         por_remodelar         6  6463980.0  5487805.0  8900000.0
ditaires_itagui          obra_negra            3  5625000.0  5396825.0  6428571.0
                         para_vivir           13  6901639.0  6060606.0  7661290.0
                         por_remodelar         2  5400538.0  4833333.0  5967742.0

## Paso 14. Limpiar los inmuebles de prueba y unirlos con su fuente

`inmuebles_prueba.csv` y `inmuebles_prueba_fuente.csv` comparten la columna `id_inmueble`.
Los unimos con `merge` (como un BUSCARV de Excel) y validamos:

1. Que no haya ids repetidos.
2. Que el `origen` y la `direccion` coincidan en ambos archivos.
3. Que las coordenadas sean números y caigan dentro del Valle de Aburrá.
4. Que cada inmueble exista en los anuncios (por `id_anuncio`).
5. Marcar las coordenadas **aproximadas**: `precision_coordenadas = barrio` significa que el punto es el
   centro del barrio, no la dirección exacta (varios inmuebles quedan en el mismo punto).

In [78]:
# 1. Ids repetidos
print("ids repetidos en inmuebles:", inmuebles["id_inmueble"].duplicated().sum())
print("ids repetidos en fuente:", fuente["id_inmueble"].duplicated().sum())

# Tipos numéricos
for col in ["area_m2", "lat", "lon"]:
    inmuebles[col] = pd.to_numeric(inmuebles[col], errors="coerce")

# Texto estandarizado
for col in ["zona", "municipio"]:
    inmuebles[col] = inmuebles[col].apply(normalizar_texto)
inmuebles["barrio"] = inmuebles["barrio"].apply(limpiar_barrio)
inmuebles = aplicar_estado(inmuebles, "inmuebles_prueba.csv", "id_inmueble")

# 2. Unir con la fuente. indicator=True nos dice si algún id está solo en un archivo
inm = inmuebles.merge(fuente, on="id_inmueble", how="outer",
                      suffixes=("", "_fuente"), indicator=True)
print(inm["_merge"].value_counts())

for col in ["origen", "direccion"]:
    a = inm[col].fillna("")
    b = inm[f"{col}_fuente"].fillna("")
    for _, f in inm[a != b].iterrows():
        anotar("inmuebles_prueba.csv", f["id_inmueble"], col, "no coincide con archivo fuente",
               f"{f[col]} vs {f[col + '_fuente']}", "revisar")
inm = inm.drop(columns=["origen_fuente", "direccion_fuente", "_merge"])

ids repetidos en inmuebles: 0
ids repetidos en fuente: 0
_merge
both          11
left_only      0
right_only     0
Name: count, dtype: int64


In [79]:
# 3. Coordenadas dentro del Valle de Aburrá (rango aproximado)
LAT_MIN, LAT_MAX = 6.0, 6.5
LON_MIN, LON_MAX = -75.75, -75.4
fuera = inm[~inm["lat"].between(LAT_MIN, LAT_MAX) | ~inm["lon"].between(LON_MIN, LON_MAX)]
for _, f in fuera.iterrows():
    anotar("inmuebles_prueba.csv", f["id_inmueble"], "lat/lon", "coordenada fuera del área", f"{f['lat']},{f['lon']}", "revisar")
print("coordenadas fuera del área:", len(fuera))

# 4. ¿Existe el anuncio?
inm["anuncio_encontrado"] = inm["id_anuncio"].isin(anuncios["id_anuncio"])
for _, f in inm[~inm["anuncio_encontrado"]].iterrows():
    anotar("inmuebles_prueba.csv", f["id_inmueble"], "id_anuncio", "no está en anuncios", f["id_anuncio"], "revisar")

# Bonus: comparar el área del inmueble con la del anuncio
inm = inm.merge(anuncios[["id_anuncio", "area_m2", "precio"]].rename(
                    columns={"area_m2": "area_anuncio", "precio": "precio_anuncio"}),
                on="id_anuncio", how="left")
for _, f in inm[inm["area_anuncio"].notna() & (inm["area_m2"] != inm["area_anuncio"])].iterrows():
    anotar("inmuebles_prueba.csv", f["id_inmueble"], "area_m2", "área distinta a la del anuncio",
           f"{f['area_m2']} vs {f['area_anuncio']}", "revisar")

# 5. Coordenadas aproximadas
inm["coordenada_aproximada"] = inm["precision_coordenadas"] != "cruce"
inm["comparte_coordenada"] = inm.duplicated(["lat", "lon"], keep=False)
for _, f in inm[inm["coordenada_aproximada"]].iterrows():
    anotar("inmuebles_prueba.csv", f["id_inmueble"], "lat/lon",
           "coordenada aproximada (centro del barrio)", f["fuente_coordenadas"],
           "ubicar dirección exacta antes del análisis de entorno")

inm[["id_inmueble", "barrio", "area_m2", "area_anuncio", "estado", "lat", "lon",
     "precision_coordenadas", "coordenada_aproximada", "comparte_coordenada", "anuncio_encontrado"]]

coordenadas fuera del área: 0


,id_inmueble,barrio,area_m2,area_anuncio,estado,lat,lon,precision_coordenadas,coordenada_aproximada,comparte_coordenada,anuncio_encontrado
0,INM-01,el salvador,73,73.0,por_remodelar,6.238961,-75.560963,cruce,False,False,True
1,INM-02,el salvador,57,57.0,por_remodelar,6.239080,-75.561813,cruce,False,False,True
2,INM-03,el salvador,73,73.0,por_remodelar,6.237597,-75.562759,cruce,False,False,True
3,INM-04,el salvador,71,71.0,para_vivir,6.234861,-75.560101,cruce,False,False,True
4,INM-05,bombona no. 1,67,67.0,por_remodelar,6.244064,-75.562360,barrio,True,False,True
5,INM-06,la castellana,90,90.0,para_vivir,6.240226,-75.605418,barrio,True,True,True
6,INM-07,la castellana,90,90.0,por_remodelar,6.240226,-75.605418,barrio,True,True,True
7,INM-08,laureles,92,92.0,para_vivir,6.242000,-75.595833,barrio,True,False,True
8,INM-09,ditaires,73,73.0,para_vivir,6.168608,-75.627492,barrio,True,True,True
9,INM-10,ditaires,64,64.0,obra_negra,6.168608,-75.627492,barrio,True,True,True


---
# PARTE B. Los archivos GeoJSON (transporte y equipamientos)

## Paso 15. Entender qué es un GeoJSON

Un GeoJSON es un archivo JSON (texto con llaves `{ }` y corchetes `[ ]`) que guarda **puntos en un mapa**.
Abre `transporte.geojson` en VS Code y verás esta estructura:

```
{
  "type": "FeatureCollection",
  "metadata": { "fuente": "OpenStreetMap...", ... },
  "features": [                                   ← lista de puntos
    {
      "geometry":   { "type": "Point", "coordinates": [-75.569, 6.247] },   ← [LONGITUD, LATITUD]
      "properties": { "nombre": "San Antonio", "sistema": "tranvia", "osm_id": "node/4100629928" }
    },
    ...
  ]
}
```

⚠ **Trampa clásica:** en GeoJSON las coordenadas vienen como **[longitud, latitud]**, al revés de como
las decimos normalmente (latitud, longitud). Si las inviertes, ¡los puntos caen en el océano!

Para trabajar con pandas tenemos que **"aplanar"** cada punto en una fila de tabla:

| osm_id | nombre | sistema | lat | lon |
|---|---|---|---|---|

In [80]:
import json

def geojson_a_tabla(nombre_archivo):
    with open(CARPETA_GEO / nombre_archivo, encoding="utf-8") as f:
        datos = json.load(f)                      # convierte el texto JSON en un diccionario de Python

    filas = []
    for punto in datos["features"]:               # recorremos cada punto
        fila = dict(punto["properties"])          # nombre, categoría, osm_id...
        lon, lat = punto["geometry"]["coordinates"]   # ¡primero longitud!
        fila["lat"] = lat
        fila["lon"] = lon
        filas.append(fila)

    df = pd.DataFrame(filas)
    print(f"{nombre_archivo}: {len(df)} puntos | columnas: {list(df.columns)}")
    print("   metadata:", datos.get("metadata"))
    return df, datos.get("metadata", {})

transporte_raw, meta_transporte = geojson_a_tabla("transporte.geojson")
equipamientos_raw, meta_equip = geojson_a_tabla("equipamientos.geojson")

transporte = transporte_raw.copy()
equipamientos = equipamientos_raw.copy()
transporte.head()

transporte.geojson: 42 puntos | columnas: ['nombre', 'sistema', 'osm_id', 'lat', 'lon']
   metadata: {'fuente': 'OpenStreetMap vía Overpass API', 'licencia': 'ODbL', 'fecha_osm': '2026-11-13T23:03:06Z', 'bbox': [6.14, -75.64, 6.29, -75.52]}
equipamientos.geojson: 2714 puntos | columnas: ['nombre', 'categoria', 'osm_id', 'lat', 'lon']
   metadata: {'fuente': 'OpenStreetMap vía Overpass API', 'licencia': 'ODbL', 'fecha_osm': '2026-11-13T23:03:06Z', 'bbox': [6.14, -75.64, 6.29, -75.52]}


,nombre,sistema,osm_id,lat,lon
0,San Antonio,tranvia,node/4100629928,6.247035,-75.569148
1,Estación del Metro Envigado,metro,node/13968486423,6.174722,-75.597065
2,Estación del Metro Itagüí,metro,node/13968486424,6.163001,-75.606585
3,San José,tranvia,relation/12601988,6.247324,-75.565395
4,Pabellón del agua EPM,tranvia,relation/12606519,6.245583,-75.561999


### Diagnóstico rápido

In [81]:
print("Transporte por sistema:\n", transporte["sistema"].value_counts(), "\n")
print("Equipamientos por categoría:\n", equipamientos["categoria"].value_counts(), "\n")
print("Equipamientos SIN nombre:", (equipamientos["nombre"].str.strip() == "").sum())
print("Nombres repetidos en transporte:\n", transporte["nombre"].value_counts().head(3))

Transporte por sistema:
 sistema
metro         23
tranvia        9
metrocable     9
metroplus      1
Name: count, dtype: int64 

Equipamientos por categoría:
 categoria
educacion         1018
parques            799
abastecimiento     392
salud              224
cultura            220
seguridad           61
Name: count, dtype: int64 

Equipamientos SIN nombre: 640
Nombres repetidos en transporte:
 nombre
San Antonio    2
Miraflores     2
Oriente        2
Name: count, dtype: int64


### ¿Qué encontramos?

| Archivo | Problema | Ejemplo |
|---|---|---|
| transporte | Prefijos inconsistentes en el nombre | `Estación del Metro Envigado` vs `Poblado` |
| transporte | Estaciones repetidas… pero **es correcto**: son estaciones de transferencia entre dos sistemas | San Antonio (metro + tranvía) |
| equipamientos | **640 puntos sin nombre** (casi todos parques) | — |
| equipamientos | El mismo lugar registrado 2 o 3 veces en OpenStreetMap | `Carulla` dos veces a 4 m de distancia |
| equipamientos | Errores de escritura y espacios dobles | `Intitucion educativa…`, `Carnes  Torino` |

## Paso 16. Limpiar `transporte`

1. Quitamos los prefijos ("Estación del Metro", "Estación de Metro de la"…) para que todos los nombres sean uniformes.
2. Validamos que las coordenadas estén dentro del área (`bbox`) que trae la metadata.
3. Marcamos las estaciones que pertenecen a **más de un sistema** (no se borran: son puntos de transferencia).

In [82]:
import re

PREFIJOS_ESTACION = r"^(estaci[oó]n\s+(del\s+|de\s+)?(metro|metrocable)\s+(de\s+)?)"

def limpiar_nombre_estacion(nombre):
    nombre = " ".join(str(nombre).split())
    nombre = re.sub(PREFIJOS_ESTACION, "", nombre, flags=re.IGNORECASE).strip()
    return nombre[:1].upper() + nombre[1:]          # primera letra en mayúscula

transporte["nombre_original"] = transporte["nombre"]
transporte["nombre"] = transporte["nombre"].apply(limpiar_nombre_estacion)
transporte["sistema"] = transporte["sistema"].apply(normalizar_texto)

for _, f in transporte[transporte["nombre"] != transporte["nombre_original"]].iterrows():
    anotar("transporte.geojson", f["osm_id"], "nombre", "prefijo en el nombre",
           f["nombre_original"], f"cambiado a {f['nombre']}")

# Validar coordenadas con el bbox de la metadata: [lat_min, lon_min, lat_max, lon_max]
def validar_bbox(df, meta, archivo):
    lat_min, lon_min, lat_max, lon_max = meta["bbox"]
    fuera = df[~df["lat"].between(lat_min, lat_max) | ~df["lon"].between(lon_min, lon_max)]
    for _, f in fuera.iterrows():
        anotar(archivo, f["osm_id"], "lat/lon", "punto fuera del área de estudio", f"{f['lat']},{f['lon']}", "revisar")
    print(f"{archivo}: {len(fuera)} puntos fuera del bbox")

validar_bbox(transporte, meta_transporte, "transporte.geojson")

transporte["multisistema"] = transporte.duplicated("nombre", keep=False)
transporte[transporte["nombre"] != transporte["nombre_original"]][["nombre_original", "nombre", "sistema"]]

transporte.geojson: 1 puntos fuera del bbox


,nombre_original,nombre,sistema
1,Estación del Metro Envigado,Envigado,metro
2,Estación del Metro Itagüí,Itagüí,metro
12,Estación Metrocable La Aurora,La Aurora,metrocable
13,Estación de Metro de la Estrella,La Estrella,metro
39,Estación del Metro Ayurá,Ayurá,metro


## Paso 17. Limpiar `equipamientos`

1. **Espacios dobles** en los nombres → uno solo.
2. **Sin nombre** → no los borramos (un parque sin nombre sigue siendo un parque cerca del inmueble).
   Les ponemos `Sin nombre (parques)` y una columna `tiene_nombre = False`.
3. **Duplicados**. Dos reglas:
   - **Regla A:** misma categoría y **exactamente** las mismas coordenadas → es el mismo punto
     (así atrapamos "Institución…" vs "Intitucion…", que están en el mismo lugar).
   - **Regla B:** mismo nombre (sin tildes ni mayúsculas) + misma categoría + a **menos de 50 metros** → es el mismo lugar.
     Ojo: `D1` aparece 59 veces, pero son tiendas distintas en sitios distintos; la regla de los 50 m las respeta.

Para medir distancias en metros entre dos coordenadas usamos la **fórmula de Haversine**
(calcula la distancia sobre la superficie de la Tierra). No necesitas entender la matemática:
solo saber que recibe dos puntos (lat, lon) y devuelve metros.

In [83]:
def distancia_metros(lat1, lon1, lat2, lon2):
    # Fórmula de Haversine. Funciona con números sueltos o con columnas/arreglos completos.
    R = 6_371_000  # radio de la Tierra en metros
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Prueba: de la estación Poblado a la estación Aguacatala (deberían ser ~2 km)
print(round(distancia_metros(6.21198, -75.5781233, 6.1939974, -75.5817818)), "metros")

2040 metros


In [84]:
# 1. Espacios dobles
equipamientos["nombre"] = equipamientos["nombre"].fillna("").apply(lambda t: " ".join(str(t).split()))
equipamientos["categoria"] = equipamientos["categoria"].apply(normalizar_texto)

# 2. Sin nombre
equipamientos["tiene_nombre"] = equipamientos["nombre"] != ""
equipamientos.loc[~equipamientos["tiene_nombre"], "nombre"] = (
    "Sin nombre (" + equipamientos["categoria"] + ")")
for _, f in equipamientos[~equipamientos["tiene_nombre"]].iterrows():
    anotar("equipamientos.geojson", f["osm_id"], "nombre", "punto sin nombre", "", "se conserva como 'Sin nombre'")

antes = len(equipamientos)

# 3A. Mismas coordenadas exactas + misma categoría (preferimos conservar el que SÍ tiene nombre)
equipamientos = equipamientos.sort_values("tiene_nombre", ascending=False)
dup_a = equipamientos[equipamientos.duplicated(["categoria", "lat", "lon"], keep="first")]
equipamientos = equipamientos.drop(dup_a.index)

# 3B. Mismo nombre + categoría a menos de 50 m
equipamientos["nombre_clave"] = equipamientos["nombre"].apply(normalizar_texto)
DISTANCIA_DUPLICADO_M = 50
indices_borrar = []
for _, grupo in equipamientos[equipamientos["tiene_nombre"]].groupby(["nombre_clave", "categoria"]):
    if len(grupo) < 2:
        continue
    conservados = []
    for idx, fila in grupo.iterrows():
        es_copia = any(distancia_metros(fila["lat"], fila["lon"], c["lat"], c["lon"]) < DISTANCIA_DUPLICADO_M
                       for c in conservados)
        if es_copia:
            indices_borrar.append(idx)
        else:
            conservados.append(fila)
dup_b = equipamientos.loc[indices_borrar]
equipamientos = equipamientos.drop(indices_borrar)

for regla, dups in [("mismas coordenadas", dup_a), (f"mismo nombre a < {DISTANCIA_DUPLICADO_M} m", dup_b)]:
    for _, f in dups.iterrows():
        anotar("equipamientos.geojson", f["osm_id"], "(fila completa)", f"duplicado ({regla})",
               f["nombre"], "eliminado")

validar_bbox(equipamientos, meta_equip, "equipamientos.geojson")
equipamientos = equipamientos.drop(columns="nombre_clave").sort_values("osm_id")
print(f"Equipamientos: {antes} → {len(equipamientos)}  "
      f"(regla A: {len(dup_a)} duplicados, regla B: {len(dup_b)} duplicados)")

equipamientos.geojson: 13 puntos fuera del bbox
Equipamientos: 2714 → 2685  (regla A: 5 duplicados, regla B: 24 duplicados)


## Paso 18. Juntar transporte y equipamientos en una sola tabla de "puntos de entorno"

Las dos tablas tienen casi las mismas columnas, así que las **apilamos** una debajo de la otra con `pd.concat`.
Agregamos una columna `tipo` para saber de dónde viene cada punto:

| id_punto | tipo | categoria | nombre | lat | lon |
|---|---|---|---|---|---|
| way/935503247 | transporte | metro | Poblado | 6.21 | -75.57 |
| node/339430113 | equipamiento | educacion | Santa Clara de Asissi | 6.24 | -75.60 |

In [85]:
t = transporte.rename(columns={"osm_id": "id_punto", "sistema": "categoria"})
t["tipo"] = "transporte"
t["tiene_nombre"] = True

e = equipamientos.rename(columns={"osm_id": "id_punto"})
e["tipo"] = "equipamiento"
e["multisistema"] = False

columnas_puntos = ["id_punto", "tipo", "categoria", "nombre", "tiene_nombre", "multisistema", "lat", "lon"]
puntos = pd.concat([t[columnas_puntos], e[columnas_puntos]], ignore_index=True)

print("¿id_punto único?", puntos["id_punto"].is_unique)
puntos.groupby(["tipo", "categoria"]).size()

¿id_punto único? True


tipo          categoria     
equipamiento  abastecimiento     385
              cultura            216
              educacion         1010
              parques            791
              salud              222
              seguridad           61
transporte    metro               23
              metrocable           9
              metroplus            1
              tranvia              9
dtype: int64

---
# PARTE C. Cruce espacial: ¿qué hay cerca de cada inmueble?

## Paso 19. Calcular distancias inmueble ↔ puntos

Aquí está la "unión" entre los CSV y los GeoJSON: no comparten una columna en común,
así que los unimos **por cercanía**. Para cada uno de los 11 inmuebles calculamos la distancia
a cada uno de los puntos (≈ 11 × 2.700 ≈ 30.000 distancias; Python lo hace en menos de un segundo).

- `RADIO_M`: radio de búsqueda. **El reto entrega un archivo de parámetros con este valor**; mientras tanto usamos 500 m.
- Guardamos los puntos hasta `RADIO_MAX_M` (1.000 m) para poder jugar con el radio en Power BI.

In [86]:
RADIO_M = 500        # ← cambia este valor según el archivo de parámetros del reto
RADIO_MAX_M = 1000

# Truco de numpy: convertir a columnas/filas hace que se calculen TODAS las combinaciones de una vez
dist = distancia_metros(
    inm["lat"].to_numpy()[:, None], inm["lon"].to_numpy()[:, None],     # 11 filas
    puntos["lat"].to_numpy()[None, :], puntos["lon"].to_numpy()[None, :] # ~2.700 columnas
)
dist_df = pd.DataFrame(dist, index=inm["id_inmueble"], columns=puntos["id_punto"])

# Pasar de "matriz" a tabla larga: una fila por (inmueble, punto)
entorno = (dist_df.stack().rename("distancia_m").reset_index())
entorno["distancia_m"] = entorno["distancia_m"].round(0)
entorno = entorno[entorno["distancia_m"] <= RADIO_MAX_M]
entorno["dentro_radio"] = entorno["distancia_m"] <= RADIO_M
entorno = entorno.merge(puntos[["id_punto", "tipo", "categoria"]], on="id_punto")

print(f"Relaciones inmueble-punto a ≤ {RADIO_MAX_M} m: {len(entorno)}")
entorno.head()

Relaciones inmueble-punto a ≤ 1000 m: 908


,id_inmueble,id_punto,distancia_m,dentro_radio,tipo,categoria
0,INM-01,relation/12606519,745.0,False,transporte,tranvia
1,INM-01,relation/12606520,605.0,False,transporte,tranvia
2,INM-01,relation/12616591,828.0,False,transporte,tranvia
3,INM-01,node/10695760482,780.0,False,equipamiento,cultura
4,INM-01,node/11638351870,537.0,False,equipamiento,abastecimiento


## Paso 20. Armar la ficha resumen de cada inmueble

Para cada inmueble calculamos:

1. **Estación de transporte más cercana** y a cuántos metros está (aunque esté a más de 1 km).
2. **Cuántos equipamientos de cada categoría** hay dentro del radio.
3. **Precio por m² de referencia**: la **mediana** del precio/m² de los anuncios comparables
   (misma zona y mismo estado, aptos y no atípicos). Usamos mediana y no promedio porque no se deja "jalar" por valores extremos.
4. **Confianza** según cuántos comparables hay. *Regla de ejemplo: ≥5 alta, 3–4 media, <3 baja.
   Ajústala con la "regla de confianza" del archivo de parámetros.*

In [87]:
# 1. Estación más cercana
cols_transporte = puntos.loc[puntos["tipo"] == "transporte", "id_punto"]
dist_transp = dist_df[cols_transporte]
cercana = pd.DataFrame({
    "id_inmueble": dist_transp.index,
    "id_estacion_cercana": dist_transp.idxmin(axis=1).values,
    "distancia_estacion_m": dist_transp.min(axis=1).round(0).values,
})
cercana = cercana.merge(
    puntos[["id_punto", "nombre", "categoria"]].rename(columns={
        "id_punto": "id_estacion_cercana", "nombre": "estacion_cercana", "categoria": "sistema_estacion"}),
    on="id_estacion_cercana")

# 2. Conteo por categoría dentro del radio (pivot = tabla dinámica)
conteos = (entorno[entorno["dentro_radio"]]
           .pivot_table(index="id_inmueble", columns="categoria", values="id_punto",
                        aggfunc="count", fill_value=0)
           .reindex(columns=sorted(puntos["categoria"].unique()), fill_value=0)  # todas las categorías, aunque den 0
           .add_prefix("n_")
           .reset_index())

# 3 y 4. Comparables de mercado
comparables = anuncios[anuncios["apto_comparable"] & ~anuncios["atipico_precio_m2"]]
ref = (comparables.groupby(["zona", "estado"])["precio_m2"]
       .agg(precio_m2_referencia="median", n_comparables="count")
       .reset_index())

def confianza(n):
    if pd.isna(n) or n < 3:
        return "baja"
    return "alta" if n >= 5 else "media"

ficha = (inm.merge(cercana, on="id_inmueble", how="left")
            .merge(conteos, on="id_inmueble", how="left")
            .merge(ref, on=["zona", "estado"], how="left"))

columnas_n = [c for c in ficha.columns if c.startswith("n_") and c != "n_comparables"]
ficha[columnas_n] = ficha[columnas_n].fillna(0).astype(int)
ficha["n_comparables"] = ficha["n_comparables"].fillna(0).astype(int)
ficha["confianza_mercado"] = ficha["n_comparables"].apply(confianza)
ficha["radio_m"] = RADIO_M

ficha[["id_inmueble", "barrio", "estado", "estacion_cercana", "distancia_estacion_m",
       *columnas_n, "precio_m2_referencia", "n_comparables", "confianza_mercado", "coordenada_aproximada"]]

,id_inmueble,barrio,estado,estacion_cercana,distancia_estacion_m,n_abastecimiento,n_cultura,n_educacion,n_metro,n_metrocable,n_metroplus,n_parques,n_salud,n_seguridad,n_tranvia,precio_m2_referencia,n_comparables,confianza_mercado,coordenada_aproximada
0,INM-01,el salvador,por_remodelar,Bicentenario,605.0,1,0,13,0,0,0,6,1,0,0,4407363.0,22,alta,False
1,INM-02,el salvador,por_remodelar,Bicentenario,637.0,0,1,12,0,0,0,7,1,0,0,4407363.0,22,alta,False
2,INM-03,el salvador,por_remodelar,Bicentenario,832.0,0,0,11,0,0,0,8,1,0,0,4407363.0,22,alta,False
3,INM-04,el salvador,para_vivir,Buenos Aires,1000.0,1,0,11,0,0,0,4,1,0,0,3909091.0,17,alta,False
4,INM-05,bombona no. 1,por_remodelar,Pabellón del agua EPM,174.0,4,6,38,0,0,0,6,4,1,3,4407363.0,22,alta,True
5,INM-06,la castellana,para_vivir,Universidad de Medellín,1126.0,5,3,12,0,0,0,8,1,1,0,6969697.0,7,alta,True
6,INM-07,la castellana,por_remodelar,Universidad de Medellín,1126.0,5,3,12,0,0,0,8,1,1,0,6463980.0,6,alta,True
7,INM-08,laureles,para_vivir,Estadio,1516.0,6,0,9,0,0,0,5,20,0,0,6969697.0,7,alta,True
8,INM-09,ditaires,para_vivir,Sabaneta,1743.0,0,1,5,0,0,0,1,0,1,0,6901639.0,13,alta,True
9,INM-10,ditaires,obra_negra,Sabaneta,1743.0,0,1,5,0,0,0,1,0,1,0,5625000.0,3,media,True


> ⚠ **Importante para la presentación:** 7 de los 11 inmuebles tienen coordenadas **aproximadas** (el centro del barrio).
> Sus distancias y conteos son orientativos. Por eso la columna `coordenada_aproximada` viaja hasta Power BI, para mostrarla en el tablero.

---
# PARTE D. Preparar el modelo para Power BI

## Paso 21. Diseñar las tablas (modelo estrella)

Power BI funciona mejor con **varias tablas pequeñas relacionadas** que con una tabla gigante.
Este es el modelo que vamos a exportar:

```
                 dim_zona ──────────────┐
                   │ zona               │ zona
                   ▼                    ▼
 dim_estado ──► inmuebles ◄──┐      anuncios ◄── dim_estado
   estado          │         │ id_inmueble
                   │ id_inmueble
                   ▼
           entorno_inmueble ──► puntos_entorno
                      id_punto
```

| Tabla | Una fila por… | Para qué sirve en el tablero |
|---|---|---|
| `inmuebles` | inmueble de prueba | La ficha: tarjetas, mapa, tabla resumen |
| `anuncios` | anuncio único (comparable) | Precio/m² por zona y estado, gráficos de dispersión |
| `puntos_entorno` | estación o equipamiento | Puntos en el mapa |
| `entorno_inmueble` | par inmueble–punto a ≤ 1 km | Contar qué hay cerca, filtrar por distancia |
| `dim_zona`, `dim_estado` | zona / estado | Segmentadores (filtros) que afectan varias tablas a la vez |
| `calidad` | problema encontrado | Página de calidad de datos |

In [88]:
NOMBRES_ZONA = {
    "buenos_aires_el_salvador": ("Buenos Aires – El Salvador", "Medellín"),
    "castellana_laureles": ("La Castellana – Laureles", "Medellín"),
    "ditaires_itagui": ("Ditaires", "Itagüí"),
}
dim_zona = pd.DataFrame([{"zona": z, "nombre_zona": n, "municipio_zona": m}
                         for z, (n, m) in NOMBRES_ZONA.items()])

NOMBRES_ESTADO = {"para_vivir": ("Para vivir", 1), "por_remodelar": ("Por remodelar", 2),
                  "obra_negra": ("Obra negra", 3), "sin_dato": ("Sin dato", 4)}
dim_estado = pd.DataFrame([{"estado": k, "nombre_estado": n, "orden_estado": o}
                           for k, (n, o) in NOMBRES_ESTADO.items()])

# Power BI no relaciona bien valores vacíos: el estado faltante pasa a "sin_dato"
anuncios_bi = anuncios.sort_values("fila_original")[[
    "id_anuncio", "fuente", "zona", "municipio", "barrio", "area_m2", "precio", "precio_m2",
    "habitaciones", "banos", "parqueadero", "estado", "estado_original", "tipo_precio",
    "apto_comparable", "atipico_precio_m2", "area_sospechosa", "posible_duplicado",
    "veces_repetido", "origenes", "fila_original"]].copy()
anuncios_bi["estado"] = anuncios_bi["estado"].fillna("sin_dato")
anuncios_bi["barrio"] = anuncios_bi["barrio"].fillna("sin dato")
anuncios_bi.insert(0, "id_comparable", ["CMP-" + str(i).zfill(3) for i in range(1, len(anuncios_bi) + 1)])

inmuebles_bi = ficha.drop(columns=["estado_dudoso", "area_anuncio", "precio_anuncio", "id_estacion_cercana"],
                          errors="ignore")
calidad = pd.DataFrame(registro)

tablas = {
    "inmuebles": inmuebles_bi,
    "anuncios": anuncios_bi,
    "puntos_entorno": puntos,
    "entorno_inmueble": entorno,
    "dim_zona": dim_zona,
    "dim_estado": dim_estado,
    "calidad": calidad,
}
for nombre, df in tablas.items():
    print(f"{nombre:<18} {df.shape[0]:>6} filas  {df.shape[1]:>3} columnas")

inmuebles              11 filas   37 columnas
anuncios               88 filas   22 columnas
puntos_entorno       2727 filas    8 columnas
entorno_inmueble      908 filas    6 columnas
dim_zona                3 filas    3 columnas
dim_estado              4 filas    3 columnas
calidad               785 filas    6 columnas


## Paso 22. Guardar los resultados

Guardamos de dos formas:

1. **`modelo_powerbi.xlsx`** (recomendado): un solo Excel con una hoja por tabla.
   Es lo más fácil para Power BI: los números y decimales llegan bien, sin importar si tu Windows
   está configurado en español (coma decimal) o en inglés (punto decimal).
2. **Un CSV por tabla** (separados por `;`), por si los prefieres o para revisar los cambios en GitHub.

In [89]:
CARPETA_POWERBI = CARPETA_LIMPIOS / "powerbi"
CARPETA_POWERBI.mkdir(exist_ok=True)

ruta_excel = CARPETA_POWERBI / "modelo_powerbi.xlsx"
with pd.ExcelWriter(ruta_excel, engine="openpyxl") as excel:
    for nombre, df in tablas.items():
        df.to_excel(excel, sheet_name=nombre, index=False)
        df.to_csv(CARPETA_POWERBI / f"{nombre}.csv", sep=";", index=False, encoding="utf-8-sig")

print("✅ Guardado:", ruta_excel.resolve())
for archivo in sorted(CARPETA_POWERBI.iterdir()):
    print("   ", archivo.name)

✅ Guardado: C:\Users\marangoh2\OneDrive - Universidad EAFIT\Personal\Estud-IA\Ficha-automatica-de-mercado-y-entorno-para-evaluar-inmuebles-en-Medellin-1\datos\limpios\powerbi\modelo_powerbi.xlsx
    anuncios.csv
    calidad.csv
    dim_estado.csv
    dim_zona.csv
    entorno_inmueble.csv
    inmuebles.csv
    modelo_powerbi.xlsx
    puntos_entorno.csv


## Paso 23. Resumen de calidad (antes vs. después)

Este resumen es ideal para presentar en la hackatón.

In [90]:
print(f"Anuncios:       {len(anuncios_raw):>5} → {len(anuncios_bi):>5} únicos")
print(f"Equipamientos:  {len(equipamientos_raw):>5} → {len(equipamientos):>5}")
print(f"Transporte:     {len(transporte_raw):>5} → {len(transporte):>5}")
print(f"Inmuebles:      {len(inmuebles_raw):>5} → {len(inmuebles_bi):>5} (con ficha de entorno y mercado)")
print()
calidad.groupby(["archivo", "problema"]).size().sort_values(ascending=False)

Anuncios:         117 →    88 únicos
Equipamientos:   2714 →  2685
Transporte:        42 →    42
Inmuebles:         11 →    11 (con ficha de entorno y mercado)



archivo                problema                                                    
equipamientos.geojson  punto sin nombre                                                640
anuncios_muestra.csv   dato faltante                                                    45
                       duplicado                                                        29
equipamientos.geojson  duplicado (mismo nombre a < 50 m)                                24
                       punto fuera del área de estudio                                  13
anuncios_muestra.csv   posible duplicado en otro portal (misma zona, área y precio)      7
inmuebles_prueba.csv   coordenada aproximada (centro del barrio)                         7
equipamientos.geojson  duplicado (mismas coordenadas)                                    5
transporte.geojson     prefijo en el nombre                                              5
anuncios_muestra.csv   valor no numérico                                                 3
      

## Próximos pasos

1. Abrir `datos/limpios/powerbi/modelo_powerbi.xlsx` en **Power BI Desktop** (ver la guía `README.md`).
2. **Validar** `MAPA_ESTADO`, `CORRECCIONES_BARRIO`, `RADIO_M` y la regla de confianza con el archivo de parámetros del reto.
3. **Comparar** la ficha con el "cálculo manual de 3 inmuebles" que entrega la organización.